**提前安装依赖包**   

pip install -U FlagEmbedding   

或   

git clone https://github.com/FlagOpen/FlagEmbedding.git  
cd FlagEmbedding  
pip install  -e .  

# 1. BGE-M3 应用

In [1]:
# 1. Dense Embedding 密集嵌入

from FlagEmbedding import BGEM3FlagModel

model = BGEM3FlagModel(model_name_or_path="/root/autodl-tmp/models/bge-m3",
                       use_fp16=True)

sentences_1 = ["这个周末我计划去海边度假，享受阳光和海浪", "最新研究表明，定期运动有助于提高工作效率和创造力"]
sentences_2 = ["我期待的假期是在沙滩上，听着海浪声放松", "科技公司最近发布了一款新的智能手机，引起了广泛关注"]

embeddings_1 = model.encode(
    sentences_1,
    batch_size=12,
    max_length=1024,
)["dense_vecs"]

embeddings_2 = model.encode(
    sentences_2,
    batch_size=12,
    max_length=1024,
)["dense_vecs"]

similarity = embeddings_1 @ embeddings_2.T

print(similarity)

/root/miniconda3/envs/txtai/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


loading existing colbert_linear and sparse_linear---------
[[0.786 0.434]
 [0.437 0.475]]


In [9]:
# 2. Sparse Embedding (Lexical Weight) 稀疏嵌入（词汇权重）

from FlagEmbedding import BGEM3FlagModel
from pprint import pprint

model = BGEM3FlagModel(model_name_or_path="/root/autodl-tmp/models/bge-m3",
                       use_fp16=True)

sentences_1 = ["这个周末我计划去海边度假，享受阳光和海浪", "最新研究表明，定期运动有助于提高工作效率和创造力"]
sentences_2 = ["周末去海边，听着海浪声方式，享受阳光", "科技公司最近发布了一款新的智能手机，引起了广泛关注"]

output_1 = model.encode(sentences_1, return_dense=True, return_sparse=True, return_colbert_vecs=False)

output_2 = model.encode(sentences_2, return_dense=True, return_sparse=True, return_colbert_vecs=False)

# 查看每个token的权重
pprint(model.convert_id_to_token(output_1["lexical_weights"]))

loading existing colbert_linear and sparse_linear---------
[{'': 0.0004246,
  ',': 0.01495,
  '享受': 0.1991,
  '去': 0.1449,
  '周末': 0.2615,
  '和': 0.0832,
  '度假': 0.266,
  '我': 0.1404,
  '浪': 0.2242,
  '海': 0.1823,
  '计划': 0.2502,
  '边': 0.2206,
  '这个': 0.1544,
  '阳光': 0.1893},
 {'': 0.02544,
  ',': 0.05295,
  '创造': 0.2042,
  '力': 0.11206,
  '和': 0.0719,
  '定期': 0.2676,
  '工作': 0.1816,
  '提高': 0.1698,
  '效率': 0.2297,
  '最新': 0.1212,
  '有助于': 0.2043,
  '研究': 0.1532,
  '表明': 0.0649,
  '运动': 0.3242}]


In [10]:
# 通过词汇匹配计算得分

lexical_scores = model.compute_lexical_matching_score(output_1["lexical_weights"][0],
                                                      output_2["lexical_weights"][0])

print(lexical_scores)

0.294203519821167


In [12]:
print(model.compute_lexical_matching_score(output_1["lexical_weights"][0],
                                           output_2["lexical_weights"][1]))

0.0004811882972717285


In [13]:
# 3. Multi-Vector (ColBERT) 多向量

from FlagEmbedding import BGEM3FlagModel

model = BGEM3FlagModel("/root/autodl-tmp/models/bge-m3", use_fp16=True)

sentences_1 = ["这个周末我计划去海边度假，享受阳光和海浪", "最新研究表明，定期运动有助于提高工作效率和创造力"]
sentences_2 = ["周末去海边，听着海浪声方式，享受阳光", "科技公司最近发布了一款新的智能手机，引起了广泛关注"]

output_1 = model.encode(sentences_1, return_dense=True, return_sparse=True, return_colbert_vecs=True)

output_2 = model.encode(sentences_2, return_dense=True, return_sparse=True, return_colbert_vecs=True)

print(model.colbert_score(output_1["colbert_vecs"][0], output_2["colbert_vecs"][0]))

loading existing colbert_linear and sparse_linear---------
tensor(0.8648)


In [14]:
print(model.colbert_score(output_1["colbert_vecs"][0], output_2["colbert_vecs"][1]))

tensor(0.3693)


In [15]:
# 4. 计算文本对的分数 
# 输入文本对列表，即可得到不同方法计算出的分数。

sentences_1 = ["什么是BGE M3？", "BM25的定义"]
sentences_2 = ["BGE M3是一个支持密集检索、词汇匹配和多向量交互的嵌入模型。", 
               "BM25是一种词袋检索函数，它根据查询词汇在每个文档中出现的情况对一组文档进行排名"]

# 构建文本对
sentence_pairs = [[i,j] for i in sentences_1 for j in sentences_2]

print(sentence_pairs)

[['什么是BGE M3？', 'BGE M3是一个支持密集检索、词汇匹配和多向量交互的嵌入模型。'], ['什么是BGE M3？', 'BM25是一种词袋检索函数，它根据查询词汇在每个文档中出现的情况对一组文档进行排名'], ['BM25的定义', 'BGE M3是一个支持密集检索、词汇匹配和多向量交互的嵌入模型。'], ['BM25的定义', 'BM25是一种词袋检索函数，它根据查询词汇在每个文档中出现的情况对一组文档进行排名']]


In [17]:
pprint(model.compute_score(sentence_pairs,
                          max_passage_length=128,
                          # weights_for_different_modes(w) 用于执行加权和：
                          # w[0]*dense_score + w[1]*sparse_score + w[2]*colbert_score
                          weights_for_different_modes=[0.4, 0.2, 0.4]))

{'colbert': [0.8098986148834229,
             0.46871069073677063,
             0.48815083503723145,
             0.8087536096572876],
 'colbert+sparse+dense': [0.6411957740783691,
                          0.34197646379470825,
                          0.3368470072746277,
                          0.6339750289916992],
 'dense': [0.70166015625, 0.38623046875, 0.352783203125, 0.68212890625],
 'sparse': [0.182861328125, 0.0, 0.0023670196533203125, 0.1881103515625],
 'sparse+dense': [0.5287272334098816,
                  0.2574869692325592,
                  0.23597781360149384,
                  0.5174559950828552]}


# 2. BGE-M3 微调

训练数据应该是一个 jsonl 文件，其中每一行都是一个像这样的字典：  
```shell
{"query": str, "pos": List[str], "neg":List[str]}
```  

如果你想使用知识蒸馏，你的 jsonl 文件的每一行应该是这样的：  
```shell
{"query": str, "pos": List[str], "neg":List[str], "pos_scores": List[float], "neg_scores": List[float]}
```   

`pos_scores`是正分数列表，其中`pos_scores[i]`是查询与教师模型中的`pos[i]`之间的分数。     
`neg_scores`是负分数列表，其中`neg_scores[i]`是查询与教师模型中的`neg[i]`之间的分数。   



下面是一个简单的例子，说明如何基于BAAI/bge-m3进行统一微调（密集嵌入、稀疏嵌入和colbert）：  

```shell
# run_BGM-M3_finetune.sh

torchrun --nproc_per_node 1 \
-m FlagEmbedding.BGE_M3.run \
--output_dir /root/autodl-tmp/flagembedding/output_M3_finetuned_model \
--model_name_or_path /root/autodl-tmp/models/bge-m3 \
--train_data /root/autodl-tmp/samples/ \
--learning_rate 1e-5 \
--fp16 \
--num_train_epochs 1 \
--per_device_train_batch_size 1 \
--dataloader_drop_last True \
--normlized True \
--temperature 0.02 \
--query_max_len 64 \
--passage_max_len 256 \
--train_group_size 2 \
--negatives_cross_device \
--logging_steps 10 \
--same_task_within_batch True \
--unified_finetuning True \
--use_self_distill True
```  

